# JAX Optimization Snapshot

This notebook demonstrates the current JAX-only 2du1 fixed field-transform FT-HMC evaluation path. Historical PyTorch-vs-JAX benchmark results are kept in `presentation/jax_benchmark_summary.md`; this notebook no longer imports or executes PyTorch.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'presentation' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from nthmc.core.jax_env import bootstrap_cuda_wheel_paths, preload_cuda_wheel_libraries
bootstrap_cuda_wheel_paths(reexec=False)
preload_cuda_wheel_libraries()

import jax
import jax.numpy as jnp

from nthmc.u1.field_transform import FieldTransformation
from nthmc.u1.u1_fthmc import build_fthmc_chain
from nthmc.u1.u1_observables import action, force, plaq_mean_from_field, topo_from_field

print('JAX backend:', jax.default_backend())
print('JAX devices:', jax.devices())


## Numerical sanity

The transform starts close to identity, and the JAX force is checked against a finite-difference directional derivative on a small lattice.


In [ ]:
L = 4
beta = 3.0
key = jax.random.PRNGKey(1029)
theta = 0.1 * jax.random.normal(key, (2, L, L), dtype=jnp.float32)
direction = jax.random.normal(jax.random.PRNGKey(1331), theta.shape, dtype=theta.dtype)
direction = direction / jnp.linalg.norm(direction)
eps = 1e-3
fd = (action(theta + eps * direction, beta) - action(theta - eps * direction, beta)) / (2 * eps)
grad_dir = jnp.sum(force(theta, beta) * direction)
transform = FieldTransformation(L, model_tag='base', n_subsets=8)
theta_t = transform.field_transformation(theta)
jac = transform.compute_jac_logdet(theta[jnp.newaxis, ...])[0]
print({'force_directional': float(grad_dir), 'finite_difference': float(fd), 'abs_error': float(jnp.abs(grad_dir - fd))})
print({'transform_mean_abs_delta': float(jnp.mean(jnp.abs(theta_t - theta))), 'jac_logdet': float(jac), 'plaquette': float(plaq_mean_from_field(theta_t)), 'topology': float(topo_from_field(theta_t))})


## Compile and steady run

This compiles the complete fixed-shape 2du1 FT-HMC evaluation chain, then runs the compiled executable a second time to estimate steady-state runtime. The lattice and sample count are intentionally small so the notebook stays comfortably under five minutes.


In [ ]:
chain = build_fthmc_chain(transform, beta=beta, n_thermalization=1, n_configs=4, n_steps=1, step_size=0.05)
run_key = jax.random.PRNGKey(2026)
t0 = time.perf_counter()
compiled = chain.lower(run_key).compile()
compile_time = time.perf_counter() - t0
t1 = time.perf_counter()
first = compiled(run_key)
jax.block_until_ready(first.plaq)
first_time = time.perf_counter() - t1
t2 = time.perf_counter()
second = compiled(jax.random.PRNGKey(2027))
jax.block_until_ready(second.plaq)
steady_time = time.perf_counter() - t2
print({'compile_time_sec': compile_time, 'first_run_sec': first_time, 'steady_run_sec': steady_time})
print({'acceptance': float(second.acceptance_rate), 'plaq_mean': float(jnp.mean(second.plaq)), 'topo_std': float(jnp.std(second.topo))})
